# Proyecto: Bank Marketing Campaign

## Clasificación con Regresión Logística

### Descripción del problema

Un banco portugués está experimentando una disminución en sus ingresos y desea identificar a los clientes existentes con mayor probabilidad de contratar un depósito a largo plazo.

El objetivo es construir un modelo de clasificación que permita predecir si un cliente contratará (`yes`) o no (`no`) el depósito.

El proyecto se desarrollará siguiendo cuatro pasos:

1. Carga del conjunto de datos.
2. Análisis exploratorio de datos (EDA).
3. Construcción de un modelo de regresión logística.
4. Optimización básica del modelo.

Este notebook está pensado para un estudiante principiante.

## Librerías

Para el EDA se utilizarán:

- `pandas`
- `matplotlib`
- `seaborn`

Para Machine Learning se utilizarán componentes básicos de `scikit-learn`.

In [ ]:
import warnings

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
sns.set_theme()

# Paso 1: Carga del conjunto de datos

Primero cargaremos el archivo directamente desde la URL proporcionada.

In [ ]:
url = "https://storage.googleapis.com/breathecode/project-files/bank-marketing-campaign-data.csv"

df = pd.read_csv(url)

print("Dimensiones del dataset:", df.shape)
df.head()

## Normalización de nombres de columnas

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(".", "_", regex=False)
)

df.columns.tolist()

# Paso 2: Análisis Exploratorio de Datos (EDA)

El EDA permite comprender la estructura, calidad y comportamiento de los datos antes de construir el modelo.

## 2.1 Exploración inicial

In [ ]:
print("Número de filas:", df.shape[0])
print("Número de columnas:", df.shape[1])

display(df.head())
display(df.dtypes)
df.info()

## 2.2 Valores faltantes

In [ ]:
missing = pd.DataFrame({
    "cantidad": df.isnull().sum(),
    "porcentaje": (df.isnull().mean() * 100).round(2)
})

missing.sort_values("porcentaje", ascending=False)

## Revisión de categorías `unknown`

In [ ]:
categorical_columns_all = df.select_dtypes(include="object").columns.tolist()

unknown_counts = {}

for columna in categorical_columns_all:
    unknown_counts[columna] = (
        df[columna]
        .astype(str)
        .str.lower()
        .eq("unknown")
        .sum()
    )

pd.Series(
    unknown_counts,
    name="cantidad_unknown"
).sort_values(ascending=False)

## 2.3 Duplicados

In [ ]:
duplicados = df.duplicated().sum()

print("Cantidad de filas duplicadas:", duplicados)
print(
    "Porcentaje de duplicados:",
    round((duplicados / len(df)) * 100, 2),
    "%"
)

In [ ]:
duplicados_df = df[df.duplicated(keep=False)]
duplicados_df.head(20)

In [ ]:
df = df.drop_duplicates().copy()

print("Dimensiones después de eliminar duplicados:", df.shape)

## 2.4 Variables numéricas y categóricas

In [ ]:
numerical_columns = df.select_dtypes(include="number").columns.tolist()

categorical_columns = (
    df.select_dtypes(include="object")
    .columns
    .drop("y")
    .tolist()
)

print("Variables numéricas:")
print(numerical_columns)

print("\nVariables categóricas:")
print(categorical_columns)

## 2.5 Variable objetivo

In [ ]:
print(df["y"].value_counts())

print("\nPorcentajes:")
print(
    (df["y"].value_counts(normalize=True) * 100).round(2)
)

In [ ]:
plt.figure(figsize=(6, 4))

sns.countplot(
    data=df,
    x="y"
)

plt.title("Distribución de la variable objetivo")
plt.xlabel("¿Contrató el depósito?")
plt.ylabel("Cantidad")
plt.show()

### Conclusión

Escribe aquí si observas desbalance entre clientes que contrataron y no contrataron.

## 2.6 Análisis univariante de variables numéricas

In [ ]:
df[numerical_columns].describe().T

### Histogramas y boxplots

In [ ]:
variables_distribucion = [
    "age",
    "duration",
    "campaign",
    "previous"
]

fig, axes = plt.subplots(
    2,
    len(variables_distribucion),
    figsize=(18, 8)
)

for col, variable in enumerate(variables_distribucion):

    sns.histplot(
        data=df,
        x=variable,
        kde=True,
        ax=axes[0, col]
    )
    axes[0, col].set_title(f"Histograma de {variable}")

    sns.boxplot(
        data=df,
        x=variable,
        ax=axes[1, col]
    )
    axes[1, col].set_title(f"Boxplot de {variable}")

plt.tight_layout()
plt.show()

Los histogramas permiten observar la forma de la distribución y los boxplots ayudan a identificar dispersión y posibles valores extremos.

## 2.7 Revisión especial de `pdays`

In [ ]:
df["pdays"].value_counts().head(10)

El valor `999` se utiliza habitualmente para representar que el cliente no había sido contactado previamente.

## 2.8 Análisis univariante de variables categóricas

In [ ]:
top_n = 15

for columna in categorical_columns:

    conteo = df[columna].value_counts().head(top_n)

    porcentaje = (
        df[columna]
        .value_counts(normalize=True)
        .head(top_n) * 100
    )

    tabla = pd.DataFrame({
        "cantidad": conteo,
        "porcentaje": porcentaje.round(2)
    })

    print(f"\nVariable: {columna}")
    display(tabla)

In [ ]:
for columna in categorical_columns:

    plt.figure(figsize=(9, 4))

    order = df[columna].value_counts().index

    sns.countplot(
        data=df,
        x=columna,
        order=order
    )

    plt.title(f"Distribución de {columna}")
    plt.xlabel(columna)
    plt.ylabel("Cantidad")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 2.9 Análisis multivariante

### Conversión temporal de la variable objetivo

In [ ]:
df["target"] = df["y"].map({
    "no": 0,
    "yes": 1
})

df[["y", "target"]].head()

### Variables numéricas vs. objetivo

In [ ]:
for columna in ["age", "campaign", "previous", "euribor3m"]:

    plt.figure(figsize=(7, 4))

    sns.boxplot(
        data=df,
        x="y",
        y=columna
    )

    plt.title(f"{columna} según resultado de campaña")
    plt.xlabel("¿Contrató?")
    plt.ylabel(columna)
    plt.tight_layout()
    plt.show()

### Variables categóricas vs. objetivo

In [ ]:
for columna in categorical_columns:

    tasa = (
        df.groupby(columna, as_index=False)["target"]
        .mean()
    )

    tasa["target"] = tasa["target"] * 100

    plt.figure(figsize=(9, 4))

    sns.barplot(
        data=tasa,
        x=columna,
        y="target"
    )

    plt.title(f"Tasa de contratación por {columna}")
    plt.xlabel(columna)
    plt.ylabel("Porcentaje de contratación")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

### Matriz de correlación

In [ ]:
correlation_matrix = df[numerical_columns + ["target"]].corr()

plt.figure(figsize=(12, 8))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f"
)

plt.title("Matriz de correlación")
plt.tight_layout()
plt.show()

## 2.10 Ingeniería de características básica

In [ ]:
df["previously_contacted"] = df["pdays"].apply(
    lambda valor: "no" if valor == 999 else "yes"
)

df[["pdays", "previously_contacted"]].head()

### Decisión sobre `duration`

`duration` se eliminará del modelo principal porque la duración de una llamada solo se conoce cuando el contacto ya terminó. Si el banco quiere decidir a quién llamar antes de realizar la llamada, esta variable no estaría disponible.

## 2.11 División en train y test

In [ ]:
X = df.drop(
    columns=[
        "y",
        "target",
        "duration"
    ]
)

y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

## 2.12 Preprocesamiento

In [ ]:
numeric_features = X_train.select_dtypes(
    include="number"
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include="object"
).columns.tolist()

print("Variables numéricas:")
print(numeric_features)

print("\nVariables categóricas:")
print(categorical_features)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_features
        )
    ]
)

# Paso 3: Modelo de Regresión Logística

Comenzaremos con una regresión logística sencilla, sin optimizar hiperparámetros.

In [ ]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

model.fit(X_train, y_train)

print("Modelo entrenado correctamente.")

In [ ]:
y_pred = model.predict(X_test)

y_probability = model.predict_proba(
    X_test
)[:, 1]

## Evaluación del modelo inicial

In [ ]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "No contrata",
            "Sí contrata"
        ]
    )
)

In [ ]:
initial_metrics = pd.Series({
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(
        y_test,
        y_pred,
        zero_division=0
    ),
    "recall": recall_score(
        y_test,
        y_pred,
        zero_division=0
    ),
    "f1": f1_score(
        y_test,
        y_pred,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_test,
        y_probability
    )
})

initial_metrics

## Matriz de confusión

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=[
        "No contrata",
        "Sí contrata"
    ]
)

plt.title("Matriz de confusión - Modelo inicial")
plt.show()

## Curva ROC

In [ ]:
RocCurveDisplay.from_predictions(
    y_test,
    y_probability
)

plt.title("Curva ROC - Modelo inicial")
plt.show()

# Paso 4: Optimización básica

Se utilizarán dos mejoras sencillas:

1. `class_weight='balanced'`
2. `GridSearchCV` con pocos valores de `C`

## 4.1 Modelo balanceado

In [ ]:
balanced_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

balanced_model.fit(X_train, y_train)

balanced_pred = balanced_model.predict(X_test)
balanced_probability = balanced_model.predict_proba(
    X_test
)[:, 1]

In [ ]:
balanced_metrics = pd.Series({
    "accuracy": accuracy_score(
        y_test,
        balanced_pred
    ),
    "precision": precision_score(
        y_test,
        balanced_pred,
        zero_division=0
    ),
    "recall": recall_score(
        y_test,
        balanced_pred,
        zero_division=0
    ),
    "f1": f1_score(
        y_test,
        balanced_pred,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_test,
        balanced_probability
    )
})

balanced_metrics

## 4.2 GridSearchCV

In [ ]:
parameter_grid = {
    "classifier__C": [
        0.1,
        1,
        10
    ],
    "classifier__class_weight": [
        None,
        "balanced"
    ]
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=parameter_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

grid_search.fit(
    X_train,
    y_train
)

print("Mejores parámetros:")
print(grid_search.best_params_)

print("\nMejor F1 en validación:")
print(grid_search.best_score_)

## Evaluación del modelo optimizado

In [ ]:
optimized_model = grid_search.best_estimator_

optimized_pred = optimized_model.predict(
    X_test
)

optimized_probability = optimized_model.predict_proba(
    X_test
)[:, 1]

optimized_metrics = pd.Series({
    "accuracy": accuracy_score(
        y_test,
        optimized_pred
    ),
    "precision": precision_score(
        y_test,
        optimized_pred,
        zero_division=0
    ),
    "recall": recall_score(
        y_test,
        optimized_pred,
        zero_division=0
    ),
    "f1": f1_score(
        y_test,
        optimized_pred,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_test,
        optimized_probability
    )
})

optimized_metrics

## Comparación final

In [ ]:
comparison = pd.DataFrame({
    "Modelo inicial": initial_metrics,
    "Modelo balanceado": balanced_metrics,
    "Modelo optimizado": optimized_metrics
}).T

comparison

In [ ]:
comparison.plot(
    kind="bar",
    figsize=(11, 5)
)

plt.title("Comparación de modelos")
plt.ylabel("Valor de la métrica")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(
    title="Métrica",
    bbox_to_anchor=(1.05, 1)
)
plt.tight_layout()
plt.show()

# Conclusiones

Completa esta sección después de ejecutar el notebook.

Puedes responder:

1. ¿La variable objetivo estaba balanceada?
2. ¿Qué observaste en `age`, `duration`, `campaign` y `previous`?
3. ¿Qué categorías mostraron mayores tasas de contratación?
4. ¿Qué variables numéricas mostraron mayor relación con `target`?
5. ¿Por qué se eliminó `duration`?
6. ¿Qué resultados obtuvo el modelo inicial?
7. ¿Qué ocurrió con `precision`, `recall` y `F1` al balancear?
8. ¿Cuál modelo elegirías?
9. ¿Cómo podría ayudar el modelo al banco?